# Go Game Review Analyser

Date: `2026-03-25`

Author: `@sthornewillve`

This notebook is the prototype that takes go games analysed and creates a database from them in order to analyse.

___

# Setup Notebook

In [1]:
import os
import json
import pandas as pd
import sqlite3

from openai import OpenAI
from go_game_analyser_helperfuncs import parse_game_reviews, summarise_game_reviews

In [2]:
# Create OpenAI Connection
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Import prompts
with open("prompt_configs.json") as f:
    prompts = json.load(f)

In [3]:
game_review_data, game_review_df = parse_game_reviews()

print(f"{len(game_review_data)} game(s) found.")
game_review_df.head()

100%|██████████| 10/10 [00:00<00:00, 8216.07it/s]

10 game(s) found.


,date,opponents_name,server,game_link,result,played_as,handicap,time_setting,review_notes
0,2026-03-27,yuki22,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+R,White,0,20 + 5x30s,\n* End of game notes\n\t* Black managed to ki...
1,2026-03-27,qp1029,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+R,White,1,20m + 5x30s,\n* End of game notes\n\t* White resigns too e...
2,2026-03-24,dienfoonwu,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+R,White,3,20m + 5x30s,\n* End of game notes\n\t* Got surrounded and ...
3,2026-03-26,xiaoqiang2026,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+9.5,White,1,20m + 5x30s,\n* End of game notes\n\t* Both players manage...
4,2026-03-26,noldoran,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+2.5,White,1,10m + 10s/mv,\n* End of game notes\n\t* Moyo game\n\t* Fair...


___

# Analyse Games

In [4]:
game_summaries = summarise_game_reviews(game_review_data, client, prompts, db_path="game_reviews.db")
print(f"{len(game_summaries)} new game(s) to analyse.")

100%|██████████| 10/10 [00:00<00:00, 221920.85it/s]

0 new game(s) to analyse.


In [5]:
game_review_df = game_review_df.loc[list(game_summaries.keys())].reset_index(drop=True)

for aspect in ["key_mistake", "key_mistake_cause", "positive_point", "game_tags"]:
    game_review_df[aspect] = [i[aspect] for i in game_summaries.values()]

game_review_df

,date,opponents_name,server,game_link,result,played_as,handicap,time_setting,review_notes,key_mistake,key_mistake_cause,positive_point,game_tags


In [6]:
if game_review_df.empty:
    print("No new games to write.")
else:
    conn = sqlite3.connect("game_reviews.db")
    game_review_df.to_sql("reviews", conn, if_exists="append", index=False)
    conn.close()
    print(f"{len(game_review_df)} new game(s) written to game_reviews.db.")

No new games to write.



___
# Appendix
## Using OpenAI API

Note that `messages` should be in the form of a list of dictionaries that has the keys `'role'` and `'content'`. Where `'role'` can be either...

1. `'system'`: The system prompt, defining the task and role for the LLM.
2. `'user'`: The actual input that you want it to process.

Be sure to be very specific about the output, everything that's not specified will be done randomly. It's usually a good idea to verify the output of the LLM and try again if it doesn't work. 

This is an example dictionary to use...

```python
messages = [
    {"role": "system", 
     "content": "You are a.../n\
     \
     Input: State input.../n\
     \
     Task: State task.../n\
     \
     Output: The Exact output..."},
    
    {"role": "user", 
     "content": "Input 1:\n...\n\n\
     \
     Input 21:\n...\n\n\
     \
     Return as described in the system message."}
]
```